# E4 — TFM contro boosting

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emanbuc/corso_foundation_tabular/blob/main/notebooks/usare-tfm/E4_tfm_vs_boosting.ipynb)

**Corso 1 · Usare un TFM — Modulo 2 (laboratorio, 20 min)**

Si confronta il TFM con LightGBM al crescere dei dati disponibili, misurando sia la qualità sia il tempo.

**Idea da far emergere:** la curva di apprendimento di TFM, LightGBM di default e LightGBM con 20 trial di tuning, e quanto costa ciascuno in tempo totale.

- **Dati:** Adult (OpenML): si predice se il reddito di una persona supera i 50.000 dollari. Variabili numeriche e categoriche, con valori mancanti.
- **Contesto:** sotto-campioni da 100 a 10.000 righe.

Serve la GPU. Senza GPU ridurre `DIMENSIONI` a `[100, 300, 1000]`.

## 0. Setup

In [ ]:
%pip install -q tabicl==2.2.0 lightgbm optuna

In [ ]:
import time

import matplotlib.pyplot as plt
import optuna
import pandas as pd
import torch
from lightgbm import LGBMClassifier
from sklearn.datasets import fetch_openml
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from tabicl import TabICLClassifier

optuna.logging.set_verbosity(optuna.logging.WARNING)
print("GPU disponibile:", torch.cuda.is_available())

## 1. I dati

Il test set è fissato una volta sola: 5.000 righe che nessun modello vede come contesto. Il resto è la riserva da cui si estraggono i contesti.

In [ ]:
adult = fetch_openml("adult", version=2, as_frame=True)
X = adult.data
y = adult.target.astype(str).str.contains(">50K").astype(int).to_numpy()

print("Righe e colonne:", X.shape)
print(f"Quota della classe positiva: {y.mean():.1%}")
print("Colonne categoriche:", list(X.select_dtypes("category").columns))

X_riserva, X_te, y_riserva, y_te = train_test_split(
    X, y, test_size=5000, stratify=y, random_state=0)

In [ ]:
def contesto(n, seed=0):
    """Un sotto-campione stratificato di n righe della riserva."""
    X_ctx, _, y_ctx, _ = train_test_split(
        X_riserva, y_riserva, train_size=n, stratify=y_riserva, random_state=seed)
    return X_ctx, y_ctx

## 2. I tre concorrenti

Tutti ricevono la stessa tabella, senza pre-elaborazione: sia `tabicl` sia LightGBM gestiscono da soli colonne categoriche e valori mancanti.

- **TFM:** nessun iperparametro.
- **LightGBM di default.**
- **LightGBM con tuning:** 20 trial di Optuna, valutati su un quarto del contesto tenuto da parte; poi il modello migliore si riaddestra su tutto il contesto. Il tempo del tuning conta nel tempo totale.

In [ ]:
def tfm(X_ctx, y_ctx):
    modello = TabICLClassifier().fit(X_ctx, y_ctx)
    return modello.predict_proba(X_te)[:, 1]


def lightgbm_default(X_ctx, y_ctx):
    modello = LGBMClassifier(random_state=0, verbose=-1).fit(X_ctx, y_ctx)
    return modello.predict_proba(X_te)[:, 1]


def lightgbm_tuning(X_ctx, y_ctx, trial=20):
    X_a, X_v, y_a, y_v = train_test_split(
        X_ctx, y_ctx, test_size=0.25, stratify=y_ctx, random_state=0)

    def obiettivo(t):
        parametri = {
            "n_estimators": t.suggest_int("n_estimators", 50, 500),
            "learning_rate": t.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "num_leaves": t.suggest_int("num_leaves", 4, 64),
            "min_child_samples": t.suggest_int("min_child_samples", 2, 50),
            "subsample": t.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_lambda": t.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        }
        modello = LGBMClassifier(**parametri, subsample_freq=1, random_state=0, verbose=-1)
        modello.fit(X_a, y_a)
        return roc_auc_score(y_v, modello.predict_proba(X_v)[:, 1])

    studio = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
    studio.optimize(obiettivo, n_trials=trial)
    modello = LGBMClassifier(**studio.best_params, subsample_freq=1, random_state=0, verbose=-1)
    modello.fit(X_ctx, y_ctx)
    return modello.predict_proba(X_te)[:, 1]


CONCORRENTI = {
    "TFM (TabICLv2)": tfm,
    "LightGBM di default": lightgbm_default,
    "LightGBM con tuning": lightgbm_tuning,
}

## 3. La curva di apprendimento

Per ogni dimensione del contesto e per ogni concorrente si misurano ROC-AUC sul test e tempo totale, cioè tutto ciò che serve per arrivare alle predizioni.

La prima chiamata al TFM scarica i pesi: la cella qui sotto la fa a vuoto, per non sporcare i tempi.

In [ ]:
_ = tfm(*contesto(100))   # riscaldamento: scarica i pesi e inizializza la GPU

In [ ]:
DIMENSIONI = [100, 300, 1000, 3000, 10000]

righe = []
for n in DIMENSIONI:
    X_ctx, y_ctx = contesto(n)
    for nome, funzione in CONCORRENTI.items():
        inizio = time.perf_counter()
        proba = funzione(X_ctx, y_ctx)
        tempo = time.perf_counter() - inizio
        auc = roc_auc_score(y_te, proba)
        righe.append({"modello": nome, "contesto": n, "ROC-AUC": auc, "tempo (s)": tempo})
        print(f"{n:6d} righe  {nome:22s} ROC-AUC = {auc:.4f}  tempo = {tempo:6.1f} s")

risultati = pd.DataFrame(righe)

In [ ]:
auc = risultati.pivot(index="contesto", columns="modello", values="ROC-AUC")[list(CONCORRENTI)]
tempi = risultati.pivot(index="contesto", columns="modello", values="tempo (s)")[list(CONCORRENTI)]
auc.round(4)

**Grafico da commentare:** a sinistra la qualità, a destra il costo. Entrambi gli assi orizzontali sono logaritmici.

In [ ]:
fig, (sx, dx) = plt.subplots(1, 2, figsize=(11, 4))
auc.plot(ax=sx, marker="o", logx=True)
sx.set_xlabel("righe di contesto")
sx.set_ylabel("ROC-AUC sul test")
sx.set_title("Curva di apprendimento")
tempi.plot(ax=dx, marker="o", logx=True, logy=True)
dx.set_xlabel("righe di contesto")
dx.set_ylabel("tempo totale (s)")
dx.set_title("Tempo per arrivare alle predizioni")
plt.tight_layout()
plt.show()

## Da osservare

- Con poche righe, quale modello è davanti? Il distacco si riduce al crescere del contesto?
- Quanto guadagna LightGBM dal tuning, e quanto tempo costa quel guadagno?
- Come cresce il tempo del TFM con le righe di contesto, rispetto a quello di LightGBM?

Una cautela: ogni punto della curva viene da un solo sotto-campione. Con 100 o 300 righe la differenza tra due modelli può dipendere dal campione estratto: per una conclusione solida si ripete con più valori di `seed` in `contesto` e si guarda la media.

## Esercizio

Trovare la dimensione del contesto oltre la quale LightGBM con tuning raggiunge il TFM, se accade.

Suggerimenti:

- la riserva contiene più di 40.000 righe: LightGBM si può addestrare anche su 20.000 o 40.000;
- il TFM su T4 va tenuto entro 10.000 righe di contesto: oltre, il suo punto di confronto resta quello a 10.000;
- la domanda pratica è: avendo 40.000 righe, conviene il boosting su tutte o il TFM su un quarto?

In [ ]:
# TODO: misurare lightgbm_tuning con contesti da 20.000 e 40.000 righe

# TODO: confrontare con la ROC-AUC del TFM a 10.000 righe